# Making Terrible Plots
Author: Brad Sheese

---

NOTE: If you want to save your work select the following from the menu above:

**File  --> Save a copy in Drive**

If you work on the copy you made all changes will be saved to Google Drive. If you do not save your own copy, you can work in 'playground mode' instead, and any changes you make will be lost after you're done.

---
##  Introduction

This is a review of different data aggregation and plotting methods.


In [ ]:
#@title Getting started? Execute this cell first.
#@markdown Click on the arrow to the left to execute this cell.

# set exercise name here
exercise_name = '09.0.4 Vis - Terrible Plots'

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import sys
import math
import urllib.request as request
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

tool_path = '/content/cs125_tools/'

if not os.path.exists(tool_path):
  ! git clone --quiet https://github.com/bsheese/cs125_tools.git

if tool_path not in sys.path:
  sys.path.append(tool_path)

import exercise_report_response

exercise_report_response.exercise_time_difficulty_report(exercise_name)


# exercise specific code starts here

url_root = 'https://raw.githubusercontent.com/'
url_repo = 'bsheese/CSDS125ExampleData/master/'
url_files = ['data_corgis_broadway.csv', 'data_corgis_airlines.csv', 'data_corgis_state_crime.csv']

for f_name in url_files:
  filedata = request.urlopen(url_root + url_repo + f_name).read().decode()
  f = open(f_name, "w+")
  f.write(filedata)
  f.close()

print()

if os.path.exists('/content/' + url_files[0]):
  print('Files successfully downloaded')
else:
  print('Something went wrong. Please ask for assistance.')


# this will change how dataframes are displayed
# %load_ext google.colab.data_table

# ignore this code entirely
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# controls plot size and dpi
# plt.rcParams['figure.figsize'] = [6, 3]
# plt.rcParams['figure.dpi'] = 150

# read the data and create the dataframe
df = pd.read_csv("data_corgis_broadway.csv", parse_dates=["Full"])

# dataset only has partial data for 1990 and 2016, drop to make the plots cleaners
df = df[df['Year'] != 1990]
df = df[df['Year'] != 2016]

if 'Full' in df.columns:
  df = df.drop('Full', axis=1)


#check dataframe
print("Corgis Broadway data loaded as 'df'.")

***
## Making Terrible Scatterplots
We can examine the associations between two continuous variables using a scatterplot. The important thing here is to make sure both variables are actually continuous. If they are not, the scatter plot will give you an error or you'll get a scatter plot with strange 'banding'(lines running either horizontally or vertically through the scatter).

The plot below is okay. Let's make it a bad plot to see what that looks like. Replace `'Gross'` with a categorical variable like 'Type' and then look at the result. Notice that the result is a non-sensical as a scatter plot.

In [ ]:
sns.scatterplot(x = 'Attendance',
                y = 'Gross',
                data = df)
plt.xlabel("Attendance")
plt.ylabel("Gross Revenue")
plt.title("Attendance vs Gross Revenue")
plt.show()

So while we are making non-sensical scatterplots, let's take a minute to scatterplot the relation between year and month (see below). Notice, Python and Pandas take no issue with doing this. They are not going to stop you for generating this sort of thing. It's going to be up to you to stop and think about what you want out a visualization and what the visualization actually means.


In [ ]:
sns.scatterplot(x = 'Year',
                y = 'Month',
                data = df)
plt.ylabel('Month')
plt.xlabel('Year')
plt.title('Month and Year Scatter Plot')
plt.show()

### Problem 1: My Ugly Scatterplot
Your turn. Generate a scatterplot that examines the relation between 'Performances' and 'Year'. It will look terrible.

In [ ]:
#enter and test your code here


In [ ]:
#@title After you've made your own terrible plot, run this to compare with mine.
sns.scatterplot(x = "Year",
                y = "Performances",
                data = df)
plt.ylabel("Performances")
plt.xlabel("Year")
plt.title("Examining Performances by Year")
plt.show()

Okay, hopefully you're looking at the plots above thinking, "we can do better than that." Try to think about what kind of plot might be appropriate to examine performance by year, then try one or two approaches to generating that plot below.

In [ ]:
# enter and test your code here


## Fixing Your Axes

Notice that when we've been working with things like Total Gross Revenue the values are so large that matplotlib is using scientific notation.

* Is high school (and scientific notation) a distant memory? [Watch this short video for a refresher](https://www.youtube.com/watch?v=DXTuYjPDjqQ).

Look back at some of the plots above. You might see something like '1e9' at the top of the y-axis or somewhere along the x-axis. That's indicating that the axis values are in scientific notation, which doesn't make for an easily interpretible visualization for most people. How would we fix this? Let's divide revenue and attendance into millions and see if that looks better.

In [ ]:
# create a new series with gross in millions
df.loc[:, 'Gross_Millions'] = df.loc[:, 'Gross']/1000000

# create a grouped dataframe
df_grouped_year_sum = df.groupby(by='Year').sum().reset_index()

# create the plot
sns.lineplot(x = 'Year',
             y = 'Gross_Millions',
             data = df_grouped_year_sum)
plt.xlabel("Year")
plt.ylabel("Total Gross Revenue in Millions")
plt.title("Total Gross Revenue by Year")
plt.show()

### Problem 2: Removing Scientific Notation
Your turn. Plot total attendance by year using `df_grouped_year_sum`. Make sure you work with the data to create a plot without any scientific notation.

In [ ]:
# enter and test your code here


In [ ]:
#@title After creating your plot, run this to compare
# create a new series with gross in millions
df_grouped_year_sum['Attendance_Millions'] = df_grouped_year_sum['Attendance']/1000000

# create the plot
sns.lineplot(x = "Year", y = "Attendance_Millions", data=df_grouped_year_sum)
plt.xlabel("Year")
plt.ylabel("Total Attendance in Millions")
plt.title("Total Attendance by Year")
plt.show()

## Making Terrible Bar Charts

You might produce bar charts that looks like the one below. The problem here is that there are many many different shows, far too many for each to have a bar. We need to either abandon this idea entirely, or restrict the number of groups bars to something manageable.

![alt text](https://github.com/bsheese/CSDS125ExampleData/raw/master/figure_plotting_toomanygroups.png)

### Better Basic Bar Charts
How we proceed to try to make a better plot depends on our research question. However, any path forward will likely require you to sub-set your data in some way.

Here's one approach: Let's say our research question was really what are the top grossing productions in the dataset? For that, let's just create a bar graph for the top five highest grossing shows.

In [ ]:
# use groupby to aggregate over weeks, move Name out of index into a column
df_grouped_name_sum = df.groupby(by='Name').sum().reset_index()

# use sort_values to sort the dataframe by Gross
df_grouped_name_sum = df_grouped_name_sum.sort_values(by='Gross_Millions', ascending=False)
df_top5 = df_grouped_name_sum[0:5] # notice the indexing here

#create plot
sns.barplot(x = 'Name',
            y = 'Gross_Millions',
            data = df_top5)
plt.ylabel("Gross Revenue in Millions($)")
plt.title("Top Five Shows by Gross Revenue")
plt.show()

That's a bit better, but still needs work. The 'Phantom of the Opera' is taking up too much space on the x-axis.

In [ ]:
# try the plot again
sns.barplot(x = 'Name',
            y = 'Gross_Millions',
            data = df_top5)
plt.ylabel("Gross Revenue in Millions($)")
plt.title("Top Five Shows by Gross Revenue")
plt.xticks(rotation=45, horizontalalignment = 'right') # rotates and aligns x-axis labels
plt.show()

### Horizontal Bar Charts for Large Numbers of Groups
Using the same process, it's easy to expand this to show the top 20 instead.

In [ ]:
# move name out of the index so we can follow the plotting conventions we've introduced so far
plt.barh(y='Name', width='Gross_Millions', data = df_grouped_name_sum[0:20])
plt.xlabel("Gross Revenue in millions($)")
plt.title("Top Twenty Shows by Gross Revenue")
plt.show()

Now, that's fairly ugly for lots of reasons. Let's reverse the sort and grab the bottom of the sorted dataframe for plotting.

In [ ]:
# change default plot size
plt.rcParams['figure.figsize'] = [5, 8]

# reverse the sort_values to sort the dataframe by Gross
df_grouped_name_sum = df_grouped_name_sum.sort_values(by='Gross_Millions', ascending=True)

# move name out of the index so we can follow the plotting conventions we've introduced so far
plt.barh(y='Name', width='Gross_Millions', data = df_grouped_name_sum[-20:])
plt.xlabel("Gross Revenue in millions($)")
plt.title("Top Twenty Shows by Gross Revenue")
plt.show()

That's a bit better.

***

In [ ]:
#@title Completed the exercise?
#@markdown Click on the arrow to the left to create the response form.

import sys
if 'exercise_report_response' not in sys.modules:
  print('Please execute the "Getting Started" cell at the beginning of this exercise, then rerun this cell.')
else:
  exercise_report_response.display_form(exercise_report_response.er_question_list)

In [ ]:
#@title Ready to Submit?
#@markdown If you have completed the exercise and the exercise response:
#@markdown * check the box 'Ready' box below
#@markdown * execute this cell

Ready = False #@param {type:"boolean"}

try:
  if Ready == True:
    exercise_report_response.submit_exercise_response(exercise_name, exercise_report_response.er_question_list)

  else:
    print('NOT SUBMITTED!!!')
    print('Check the ready box if you are ready to submit your response.')

except:
  print('NOT SUBMITTED!!!')
  print('Execute the cell above to create the form.')